# NIFTY Gap Strategy — v11 (Regime-Gated Walk-Forward)

**Key fix over v7/v8/v10:** The regime gate is baked into the training loop, not bolted on after.

| Version | Problem |
|---|---|
| v7/v8 | Model trained on ALL days including high-VIX chaos → threshold diluted |
| v10 | Gate applied after model scoring → model still saw bad-regime days in training |
| **v11** | **Gate applied before training → model only learns signal/outcome in calm regimes** |

**Workflow:**
1. Load `v6/sim_cache.csv` (SL=15%, TP=40% — original, proven SL/TP)
2. Compute `VIX_INDIA_pct` and `gap_normalized` regime features
3. Apply VIX pct gate (`≤ 0.65`) to training data before fitting model
4. Train L1 logistic regression on gate-filtered training rows only
5. OOS: gate first → score → trade if prob ≥ threshold

| Parameter | Value |
|---|---|
| SL | −15% of entry premium |
| TP | +40% of entry premium |
| Breakeven | 27.3% |
| Primary gate | VIX_INDIA_pct ≤ 0.65 (calm regime only) |
| Secondary gate | \|gap_normalized\| ∈ [0.10, 0.80] (optional, tested separately) |
| Train | 2024 – Jun 2025 (gate-filtered) |
| OOS | Jul 2025+ (gate → score → trade) |

In [1]:
import warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import date
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Trade parameters (same as v6/v7) ─────────────────────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
BREAKEVEN        = SL_PCT / (SL_PCT + TP_PCT)   # 27.3%
KELLY_ODDS       = TP_PCT / SL_PCT               # 2.667

# ── Walk-forward split ────────────────────────────────────────────────────────
TRAIN_END = date(2025, 6, 30)
OOS_START = date(2025, 7, 1)

# ── Regime gate thresholds ────────────────────────────────────────────────────
VIX_PCT_MAX  = 0.65   # primary gate: India VIX ≤ 65th rolling percentile
GAP_NORM_MAX = 0.80   # secondary gate: |gap_normalized| ≤ this
GAP_NORM_MIN = 0.10   # secondary gate: |gap_normalized| ≥ this

# ── Model config ─────────────────────────────────────────────────────────────
MIN_THRESH_TRADES = 10
EDGE_TARGET_PP    = 8

FEATURES = [
    'gap_pct', 'prev_india_ret', 'us_ret', 'europe_ret', 'asia_ret',
    'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level', 'log_entry_prem', 'dte',
    'nifty_20d_ret', 'nifty_20d_realized_vol', 'gap_normalized',
]

print('Config loaded.')
print(f'SL={SL_PCT:.0%}  TP={TP_PCT:.0%}  Breakeven={BREAKEVEN:.1%}')
print(f'Train: 2024–{TRAIN_END}  |  OOS: {OOS_START}+')
print(f'Primary gate: VIX_INDIA_pct ≤ {VIX_PCT_MAX}')
print(f'Features: {len(FEATURES)}')

Config loaded.
SL=15%  TP=40%  Breakeven=27.3%
Train: 2024–2025-06-30  |  OOS: 2025-07-01+
Primary gate: VIX_INDIA_pct ≤ 0.65
Features: 13


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING    = Path.cwd().parent
SIM_CACHE_PATH = GAP_TRADING / 'v6' / 'sim_cache.csv'
ALIGNED_CSV    = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'

for lbl, p in [('sim_cache', SIM_CACHE_PATH), ('aligned CSV', ALIGNED_CSV)]:
    print(f'{lbl:<12}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load sim_cache ────────────────────────────────────────────────────────────
sim_df = pd.read_csv(SIM_CACHE_PATH, parse_dates=['date'])
sim_df['date'] = sim_df['date'].dt.date
sim_core = sim_df[['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte']].copy()
print(f'\nsim_cache : {len(sim_df)} rows  ({sim_df["date"].min()} → {sim_df["date"].max()})')
print(f'Overall WR: {sim_df["win"].mean():.1%}')

# ── Load aligned + rolling features ──────────────────────────────────────────
aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned['india_date'] = aligned['india_date'].dt.date
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()
aligned = aligned.sort_values('india_date').reset_index(drop=True)

aligned['nifty_20d_realized_vol'] = aligned['prev_india_ret'].rolling(20).std()
aligned['nifty_20d_ret'] = (
    np.exp(np.log1p(aligned['prev_india_ret']).rolling(20).sum()) - 1
)
aligned['VIX_INDIA_pct'] = (
    aligned['VIX_INDIA_level']
    .rolling(252, min_periods=60)
    .rank(pct=True)
)
print(f'aligned   : {len(aligned)} rows')

# ── Merge ─────────────────────────────────────────────────────────────────────
ALIGNED_COLS = [
    'india_date', 'gap_pct', 'prev_india_ret',
    'SP500_ret', 'NASDAQ_ret', 'DOW_ret',
    'DAX_ret', 'FTSE_ret',
    'NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret',
    'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level',
    'nifty_20d_ret', 'nifty_20d_realized_vol', 'VIX_INDIA_pct',
]
merged = (
    sim_core
    .merge(aligned[ALIGNED_COLS], left_on='date', right_on='india_date', how='inner')
    .drop(columns=['india_date'])
)
merged['us_ret']         = merged[['SP500_ret', 'NASDAQ_ret', 'DOW_ret']].mean(axis=1)
merged['europe_ret']     = merged[['DAX_ret', 'FTSE_ret']].mean(axis=1)
merged['asia_ret']       = merged[['NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret']].mean(axis=1)
merged['log_entry_prem'] = np.log(merged['entry_prem'].clip(lower=0.1))
merged['gap_normalized'] = merged['gap_pct'] / merged['nifty_20d_realized_vol'].replace(0, np.nan)
merged['quarter']        = merged['date'].apply(lambda d: f'{d.year}Q{(d.month-1)//3+1}')

# Drop rolling warmup NaNs
gate_cols = ['gap_normalized', 'VIX_INDIA_pct', 'nifty_20d_realized_vol']
before = len(merged)
merged = merged.dropna(subset=gate_cols).reset_index(drop=True)
merged = merged.sort_values('date').reset_index(drop=True)
print(f'merged    : {len(merged)} rows  (dropped {before - len(merged)} NaN warmup rows)')

# ── Train / OOS split (full, pre-gate — for reference) ───────────────────────
train_all = merged[merged['date'] <= TRAIN_END].copy()
oos_all   = merged[merged['date'] >= OOS_START].copy()
print(f'\nTrain (all): {len(train_all)} rows  |  OOS: {len(oos_all)} rows')
print(f'Train WR (all days): {train_all["win"].mean():.1%}')

sim_cache   : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv)
aligned CSV : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)

sim_cache : 402 rows  (2024-01-02 → 2026-03-24)
Overall WR: 24.9%
aligned   : 740 rows
merged    : 402 rows  (dropped 0 NaN warmup rows)

Train (all): 269 rows  |  OOS: 133 rows
Train WR (all days): 23.8%


In [3]:
# ── Gate functions ────────────────────────────────────────────────────────────
def gate_vix(df):
    return df['VIX_INDIA_pct'] <= VIX_PCT_MAX

def gate_gap(df):
    return (df['gap_normalized'].abs() <= GAP_NORM_MAX) & \
           (df['gap_normalized'].abs() >= GAP_NORM_MIN)

def gate_both(df):
    return gate_vix(df) & gate_gap(df)

# ── Gate diagnostic: quarterly WR before and after gate ──────────────────────
print('=' * 72)
print('  QUARTERLY WIN RATE — before and after gate  (full dataset)')
print(f'  Breakeven = {BREAKEVEN:.1%}')
print('=' * 72)
print(f'{"Quarter":<9} {"N_all":>5} {"WR_all":>7}  '
      f'{"N_vix":>6} {"WR_vix":>7}  '
      f'{"N_both":>6} {"WR_both":>7}')
print('-' * 72)

for q in sorted(merged['quarter'].unique()):
    qdf   = merged[merged['quarter'] == q]
    g_vix = gate_vix(qdf)
    g_both= gate_both(qdf)
    def _wr(mask): return qdf[mask]['win'].mean() if mask.sum() > 0 else float('nan')
    flag = ' ← BAD' if _wr(pd.Series(True, index=qdf.index)) < BREAKEVEN else ''
    print(f'{q:<9} {len(qdf):>5} {_wr(pd.Series(True,index=qdf.index)):>7.1%}  '
          f'{g_vix.sum():>6} {_wr(g_vix):>7.1%}  '
          f'{g_both.sum():>6} {_wr(g_both):>7.1%}{flag}')
print('=' * 72)

# ── Gate-filtered training set ────────────────────────────────────────────────
vix_mask_train  = gate_vix(train_all)
both_mask_train = gate_both(train_all)

train_vix  = train_all[vix_mask_train].copy().reset_index(drop=True)
train_both = train_all[both_mask_train].copy().reset_index(drop=True)

print(f'\nTraining set sizes:')
print(f'  All days          : {len(train_all):>3}  WR = {train_all["win"].mean():.1%}')
print(f'  VIX gate only     : {len(train_vix):>3}  WR = {train_vix["win"].mean():.1%}  ({len(train_vix)/len(train_all):.0%} kept)')
print(f'  VIX + gap gate    : {len(train_both):>3}  WR = {train_both["win"].mean():.1%}  ({len(train_both)/len(train_all):.0%} kept)')

  QUARTERLY WIN RATE — before and after gate  (full dataset)
  Breakeven = 27.3%
Quarter   N_all  WR_all   N_vix  WR_vix  N_both WR_both
------------------------------------------------------------------------
2024Q1       47   29.8%       2    0.0%       1    0.0%
2024Q2       44   27.3%      13   30.8%      10   30.0%
2024Q3       46   17.4%      31    6.5%      20    0.0% ← BAD
2024Q4       42   26.2%      24   33.3%      18   38.9% ← BAD
2025Q1       48   27.1%      28   25.0%      18   33.3% ← BAD
2025Q2       42   14.3%      15   13.3%       9   11.1% ← BAD
2025Q3       49   22.4%      49   22.4%      34   23.5% ← BAD
2025Q4       43   27.9%      43   27.9%      32   34.4%
2026Q1       41   31.7%      24   33.3%      17   23.5%

Training set sizes:
  All days          : 269  WR = 23.8%
  VIX gate only     : 113  WR = 20.4%  (42% kept)
  VIX + gap gate    :  76  WR = 22.4%  (28% kept)


In [4]:
# ── Train two model variants: VIX-gate-only and VIX+gap-gate ─────────────────
def impute(df, cols):
    X = df[cols].copy()
    return X.fillna(X.median())

def fit_model(train_df, label):
    print(f'\n{"-"*58}')
    print(f'  Fitting: {label}  ({len(train_df)} training rows)')
    print(f'  Training WR: {train_df["win"].mean():.1%}')
    print(f'{"-"*58}')

    X_raw = impute(train_df, FEATURES)
    y     = train_df['win'].astype(int)

    scaler  = StandardScaler()
    X_scaled= scaler.fit_transform(X_raw)

    model = LogisticRegressionCV(
        penalty='l1', solver='liblinear',
        Cs=[0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
        cv=5, scoring='roc_auc',
        class_weight={0: 1, 1: 2.5},
        max_iter=500, random_state=42,
    )
    model.fit(X_scaled, y)

    train_probs = model.predict_proba(X_scaled)[:, 1]
    train_auc   = roc_auc_score(y, train_probs)
    best_C      = model.C_[0]

    print(f'  Best C    : {best_C}')
    print(f'  Train AUC : {train_auc:.3f}')

    # Non-zero coefficients
    coef_df = pd.DataFrame({'Feature': FEATURES, 'Coef': model.coef_[0]})
    nonzero = coef_df[coef_df['Coef'] != 0].sort_values('Coef', ascending=False)
    zeroed  = coef_df[coef_df['Coef'] == 0]['Feature'].tolist()
    print(f'  L1 zeroed : {zeroed}')
    print(f'  Non-zero coefficients:')
    for _, r in nonzero.iterrows():
        print(f'    {r["Feature"]:<28} {r["Coef"]:+.4f}')

    # Threshold sweep on gate-filtered training data
    train_df = train_df.copy()
    train_df['prob'] = train_probs
    base_wr = train_df['win'].mean()

    thresholds = np.arange(0.28, 0.65, 0.02).round(2)
    thr_rows = []
    for thr in thresholds:
        sub = train_df[train_df['prob'] >= thr]
        if len(sub) == 0: continue
        thr_rows.append({
            'Threshold': thr, 'Trades': len(sub),
            'Win%': round(sub['win'].mean() * 100, 1),
            'Edge_pp': round((sub['win'].mean() - base_wr) * 100, 1),
        })
    thr_df = pd.DataFrame(thr_rows)
    print(f'\n  Threshold sweep (on gate-filtered train data, base WR={base_wr:.1%}):')
    print(thr_df.to_string(index=False))

    # Auto-select threshold
    target_wr = base_wr + EDGE_TARGET_PP / 100
    candidates = thr_df[(thr_df['Win%'] >= target_wr * 100) & (thr_df['Trades'] >= MIN_THRESH_TRADES)]
    if len(candidates) > 0:
        selected_thr = float(candidates['Threshold'].iloc[-1])
    else:
        fallback = thr_df[(thr_df['Edge_pp'] > 0) & (thr_df['Trades'] >= 5)]
        selected_thr = float(fallback['Threshold'].iloc[-1]) if len(fallback) > 0 else 0.35
        print(f'  WARNING: fallback threshold used.')
    print(f'\n  Auto-selected threshold: {selected_thr}')

    bundle = {
        'model': model, 'scaler': scaler, 'threshold': selected_thr,
        'features': FEATURES, 'train_auc': round(train_auc, 3),
        'best_C': best_C, 'gate_label': label,
        'train_n': len(train_df), 'train_wr': round(base_wr, 4),
    }
    return bundle

bundle_vix  = fit_model(train_vix,  'VIX gate only')
bundle_both = fit_model(train_both, 'VIX + gap gate')

# Save both
OUT = Path.cwd()
joblib.dump(bundle_vix,  OUT / 'v11_model_vix.pkl')
joblib.dump(bundle_both, OUT / 'v11_model_both.pkl')
print(f'\nSaved: v11_model_vix.pkl, v11_model_both.pkl')


----------------------------------------------------------
  Fitting: VIX gate only  (113 training rows)
  Training WR: 20.4%
----------------------------------------------------------
  Best C    : 0.5
  Train AUC : 0.775
  L1 zeroed : ['us_ret', 'log_entry_prem']
  Non-zero coefficients:
    gap_pct                      +0.3173
    nifty_20d_realized_vol       +0.2004
    gap_normalized               +0.0645
    VIX_INDIA_level              +0.0413
    dte                          +0.0092
    VIX_US_ret                   -0.2077
    europe_ret                   -0.2378
    prev_india_ret               -0.2452
    nifty_20d_ret                -0.4206
    VIX_US_level                 -0.4646
    asia_ret                     -0.5234

  Threshold sweep (on gate-filtered train data, base WR=20.4%):
 Threshold  Trades  Win%  Edge_pp
      0.28      75  30.7     10.3
      0.30      69  31.9     11.5
      0.32      68  32.4     12.0
      0.34      64  34.4     14.0
      0.36      57  35

In [5]:
# ── Backtest engine ───────────────────────────────────────────────────────────
def round_trip_charges(ep, xp, lots):
    bv = ep * lots * LOT_SIZE; sv = xp * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * bv
    stt   = 0.000625 * sv
    exch  = 0.00053  * (bv + sv)
    sebi  = 0.000001 * (bv + sv)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)

def run_backtest(label, oos_df, bundle, gate_fn):
    model    = bundle['model']
    scaler   = bundle['scaler']
    thr      = bundle['threshold']
    base_wr  = bundle['train_wr']

    # Apply gate to OOS
    gate_mask   = gate_fn(oos_df)
    gated       = oos_df[gate_mask].copy().reset_index(drop=True)
    n_filtered  = (~gate_mask).sum()

    if len(gated) == 0:
        print(f'\n{label}: 0 rows pass gate.')
        return pd.DataFrame()

    # Score gate-passing rows
    X = impute(gated, FEATURES)
    gated['prob'] = model.predict_proba(scaler.transform(X.values))[:, 1]

    tradeable  = gated[gated['prob'] >= thr].copy()
    n_low_prob = (gated['prob'] < thr).sum()

    if len(tradeable) == 0:
        print(f'\n{label}: gate passed {len(gated)}, but 0 above threshold {thr}.')
        return pd.DataFrame()

    capital, peak, rows = STARTING_CAPITAL, STARTING_CAPITAL, []
    for _, row in tradeable.iterrows():
        ep, xp, dte = float(row['entry_prem']), float(row['exit_prem']), int(row['dte'])
        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0: continue
        lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0: lots = min(lots, DTE0_MAX_LOTS)
        chg = round_trip_charges(ep, xp, lots)
        pnl = (xp - ep) * LOT_SIZE * lots - chg
        capital += pnl; peak = max(peak, capital)
        rows.append({
            'Date': row['date'], 'Quarter': row['quarter'],
            'P(win)': round(float(row['prob']), 3), 'DTE': dte, 'Lots': lots,
            'Entry': ep, 'Exit': xp, 'PnL(pts)': round(xp - ep, 2),
            'Trade PnL': round(pnl, 2), 'Capital': round(capital, 2),
            'DD%': round((peak - capital) / peak * 100, 2),
            'Reason': row['exit_reason'],
        })

    ledger = pd.DataFrame(rows)
    wins   = (ledger['Trade PnL'] > 0).sum()
    total  = len(ledger)
    roi    = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd  = ledger['DD%'].max()
    aw = ledger.loc[ledger['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0     else 0
    al = ledger.loc[ledger['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total else 0

    print(f'\n{"="*62}')
    print(f'  {label}')
    print(f'{"="*62}')
    print(f'  OOS days total          : {len(oos_df)}')
    print(f'  Gate filtered out       : {n_filtered}  ({n_filtered/len(oos_df):.0%})')
    print(f'  Gate passed             : {len(gated)}')
    print(f'  Below threshold (p<{thr}): {n_low_prob}')
    print(f'  Traded                  : {total}')
    print(f'  Win rate : {wins/total*100:.1f}%  (train gate WR: {base_wr:.1%}  |  BE: {BREAKEVEN:.1%})')
    print(f'  ROI      : {roi:+.1f}%')
    print(f'  Max DD   : {maxdd:.1f}%')
    print(f'  Avg win  : Rs {aw:,.0f}  |  Avg loss: Rs {al:,.0f}')
    print(f'{"="*62}')
    print(ledger['Reason'].value_counts().to_string())
    print()
    print(ledger[['Date','Quarter','P(win)','DTE','Lots','Entry','Exit',
                  'PnL(pts)','Trade PnL','Capital','DD%','Reason']].to_string(index=False))
    return ledger

# ── Run OOS backtests ─────────────────────────────────────────────────────────
print('Running OOS backtests...')
ledger_vix  = run_backtest('v11 OOS — VIX gate + model (vix-trained)',  oos_all, bundle_vix,  gate_vix)
ledger_both = run_backtest('v11 OOS — VIX+gap gate + model (both-trained)', oos_all, bundle_both, gate_both)

Running OOS backtests...

  v11 OOS — VIX gate + model (vix-trained)
  OOS days total          : 133
  Gate filtered out       : 17  (13%)
  Gate passed             : 116
  Below threshold (p<0.62): 111
  Traded                  : 5
  Win rate : 40.0%  (train gate WR: 20.3%  |  BE: 27.3%)
  ROI      : +10.6%
  Max DD   : 19.1%
  Avg win  : Rs 33,723  |  Avg loss: Rs -15,387
Reason
Stop Loss     3
Target Hit    2

      Date Quarter  P(win)  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
2025-07-29  2025Q3   0.658    2    25  65.35  55.55     -9.80  -18633.01 181366.99  9.32  Stop Loss
2025-09-04  2025Q3   0.644    5    25  68.85  58.52    -10.33  -19638.04 161728.95 19.14  Stop Loss
2025-12-17  2025Q4   0.699    6    25  78.70 110.18     31.48   58622.35 220351.30  0.00 Target Hit
2026-01-20  2026Q1   0.624    0    10  29.75  41.65     11.90    8824.05 229175.35  0.00 Target Hit
2026-01-27  2026Q1   0.740    0    10  68.95  58.61    -10.34   -7891.17 221284.18 

In [6]:
# ── In-sample reference (on gate-filtered train data) ────────────────────────
ledger_train_vix  = run_backtest('TRAIN — VIX gate + model (in-sample)',     train_all, bundle_vix,  gate_vix)
ledger_train_both = run_backtest('TRAIN — VIX+gap gate + model (in-sample)', train_all, bundle_both, gate_both)


  TRAIN — VIX gate + model (in-sample)
  OOS days total          : 269
  Gate filtered out       : 156  (58%)
  Gate passed             : 113
  Below threshold (p<0.62): 102
  Traded                  : 11
  Win rate : 54.5%  (train gate WR: 20.3%  |  BE: 27.3%)
  ROI      : +74.1%
  Max DD   : 28.6%
  Avg win  : Rs 47,924  |  Avg loss: Rs -27,868
Reason
Target Hit    5
Stop Loss     5
11:15 exit    1

      Date Quarter  P(win)  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
2024-04-04  2024Q2   0.678    0    10  43.10  60.34     17.24   12804.94 212804.94  0.00 Target Hit
2024-06-19  2024Q2   0.629    1    25  56.55  79.17     22.62   42109.89 254914.83  0.00 Target Hit
2024-06-21  2024Q2   0.642    6    25 116.75 134.85     18.10   33430.12 288344.95  0.00 11:15 exit
2024-07-03  2024Q3   0.751    1    25  69.40  58.99    -10.41  -19789.82 268555.13  6.86  Stop Loss
2024-10-22  2024Q4   0.704    2    25  91.60  77.86    -13.74  -26105.18 242449.95 15.92  Stop

In [7]:
# ── Summary + export ──────────────────────────────────────────────────────────
def _row(ledger, label):
    if ledger.empty:
        return {'Strategy': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A'}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['DD%'].max()
    return {'Strategy': label, 'Trades': total,
            'Win%': f'{wins/total*100:.1f}%', 'ROI': f'{roi:+.1f}%', 'MaxDD': f'{maxdd:.1f}%'}

rows = [
    {'Strategy': 'v7  (no gate, threshold=0.58)',          'Trades': 10, 'Win%': '40.0%', 'ROI': '+8.4%',  'MaxDD': '21.0%'},
    {'Strategy': 'v8  L1 walk-forward (no gate)',          'Trades':  7, 'Win%': '28.6%', 'ROI': '-19.5%', 'MaxDD': '24.8%'},
    {'Strategy': 'v10 VIX gate (model trained on ALL)',    'Trades':  5, 'Win%': '40.0%', 'ROI': '+29.3%', 'MaxDD': '21.0%'},
    _row(ledger_vix,  'v11 VIX gate (model trained on VIX-filtered)'),
    _row(ledger_both, 'v11 VIX+gap gate (model trained on both-filtered)'),
]

summary_df = pd.DataFrame(rows)

print()
print('=' * 72)
print('  v11 SUMMARY — gate baked into training')
print('=' * 72)
print(summary_df.to_string(index=False))
print('=' * 72)
print(f'SL={SL_PCT:.0%} / TP={TP_PCT:.0%} / BE={BREAKEVEN:.1%}')
print(f'v11-vix  model: threshold={bundle_vix["threshold"]}  train_auc={bundle_vix["train_auc"]}  train_n={bundle_vix["train_n"]}  train_wr={bundle_vix["train_wr"]:.1%}')
print(f'v11-both model: threshold={bundle_both["threshold"]}  train_auc={bundle_both["train_auc"]}  train_n={bundle_both["train_n"]}  train_wr={bundle_both["train_wr"]:.1%}')

# ── Export ────────────────────────────────────────────────────────────────────
OUT = Path.cwd()
summary_df.to_csv(OUT / 'v11_summary.csv', index=False)
print(f'\nExported: v11_summary.csv')

for fname, ledger in [
    ('v11_oos_vix_gate.csv',       ledger_vix),
    ('v11_oos_both_gates.csv',     ledger_both),
    ('v11_train_vix_gate.csv',     ledger_train_vix),
    ('v11_train_both_gates.csv',   ledger_train_both),
]:
    if not ledger.empty:
        ledger.to_csv(OUT / fname, index=False)
        print(f'Exported: {fname}  ({len(ledger)} trades)')


  v11 SUMMARY — gate baked into training
                                         Strategy  Trades  Win%    ROI MaxDD
                    v7  (no gate, threshold=0.58)      10 40.0%  +8.4% 21.0%
                    v8  L1 walk-forward (no gate)       7 28.6% -19.5% 24.8%
              v10 VIX gate (model trained on ALL)       5 40.0% +29.3% 21.0%
     v11 VIX gate (model trained on VIX-filtered)       5 40.0% +10.6% 19.1%
v11 VIX+gap gate (model trained on both-filtered)      17 47.1% +88.0% 15.4%
SL=15% / TP=40% / BE=27.3%
v11-vix  model: threshold=0.62  train_auc=0.775  train_n=113  train_wr=20.3%
v11-both model: threshold=0.52  train_auc=0.687  train_n=76  train_wr=22.4%

Exported: v11_summary.csv
Exported: v11_oos_vix_gate.csv  (5 trades)
Exported: v11_oos_both_gates.csv  (17 trades)
Exported: v11_train_vix_gate.csv  (11 trades)
Exported: v11_train_both_gates.csv  (20 trades)
